In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

In [2]:
# The unique identifier for BOA Bank's app on the Google Play Store
BOA_APP_ID = 'com.boa.boaMobileBanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    BOA_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("BOA Bank App Info")
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

BOA Bank App Info
App Title   : BoA Mobile
Current Score: 4.3909855
Total Ratings: 9,215
Total Reviews: 1,460
Installs     : 1,000,000+


In [3]:
# Step 2: Scrape reviews
print(f"Scraping reviews for Awash Bank...")

result, continuation_token = reviews(
    BOA_APP_ID,
    lang='en',
    country='et',
    sort=Sort.MOST_RELEVANT,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Awash Bank...
Collected 500 raw reviews


In [4]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 94d4c6ee-67e6-454f-9744-a5c495d30d1f
  userName: Hannes Garben
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjXlwcrH9-wSIht_fPSyXNL6hU42h-xn3gEOIDxh2ikTqX-7ViVryA
  content: the app does not show my account balance correctly compared to what I get with the SMS notification. Again and again I am not able to make transfers because the app has issues. The app has over years now frequently problems, doesn't load, or is sometimes not working entirely and I can not open it. Even to the point that I question the rating of this app if it's true or distorted. I am not sure how it can be 4 stars something, when in real life the app just doesn't work a lot of the time.
  score: 1
  thumbsUpCount: 16
  reviewCreatedVersion: 26.03.13
  at: 2026-04-19 03:43:52
  replyContent: N

In [5]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,94d4c6ee-67e6-454f-9744-a5c495d30d1f,the app does not show my account balance corre...,1,2026-04-19 03:43:52,Awash Bank,Google Play
1,a5f24e46-f662-4b0c-bfd9-201a1ffae051,Personal opinion: If Abyssinia Mobile Banking ...,5,2026-03-31 02:54:21,Awash Bank,Google Play
2,a366f66f-c0a0-425e-add1-7e71b1610e2f,"Bank of Abyssinia used to be the best for me, ...",1,2026-03-21 15:37:47,Awash Bank,Google Play
3,63ebb0e3-90cf-4ac2-b914-226871e5f4eb,"I tried to oppen mobile app of BOA, but it can...",1,2026-04-30 23:20:45,Awash Bank,Google Play
4,edff7faf-b934-4fe1-b2c8-e09d9969b7c0,The worst banking app ever. When I toggle the ...,1,2026-03-19 05:36:05,Awash Bank,Google Play


In [6]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [7]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    print(f"  {int(rating)} stars: {count:>4}")

Rating distribution:
  5 stars:   98
  4 stars:   22
  3 stars:   37
  2 stars:   32
  1 stars:  311


In [8]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-04-19 03:43:52
1   2026-03-31 02:54:21
2   2026-03-21 15:37:47
3   2026-04-30 23:20:45
4   2026-03-19 05:36:05
5   2026-02-27 05:46:37
6   2026-02-22 21:12:09
7   2024-08-03 13:59:13
8   2024-05-08 05:59:05
9   2026-03-16 13:29:33

Date dtype: datetime64[us]


In [9]:
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [10]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 0
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [11]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-04-19 03:43:52
  Target format: YYYY-MM-DD (string or date object)


In [12]:
# Work on a copy so raw data stays untouched
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

Starting with: 500 reviews


In [13]:
before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


In [14]:
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Removed 0 duplicate reviews
Remaining: 500 reviews


In [15]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-04-19 03:43:52
1   2026-03-31 02:54:21
2   2026-03-21 15:37:47
dtype: datetime64[us]

After normalization:
0    2026-04-19
1    2026-03-31
2    2026-03-21
dtype: str

Date range: 2024-01-11 to 2026-05-11


In [16]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


In [17]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


In [18]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,this app is good but the speed of app is very ...,2,2026-05-11,Awash Bank,Google Play
1,boa the best,5,2026-05-08,Awash Bank,Google Play
2,extremely slow app and unreliable for most pay...,2,2026-05-03,Awash Bank,Google Play
3,"I tried to oppen mobile app of BOA, but it can...",1,2026-04-30,Awash Bank,Google Play
4,easy to understand,4,2026-04-28,Awash Bank,Google Play
5,there's an error in app i don't ever know what...,1,2026-04-24,Awash Bank,Google Play
6,what an amazing service,5,2026-04-23,Awash Bank,Google Play
7,the worst app,1,2026-04-23,Awash Bank,Google Play
8,everything new that other banks started abyssi...,1,2026-04-20,Awash Bank,Google Play
9,"the best applcation,, how can winning byd car 😭🤣🤣",5,2026-04-19,Awash Bank,Google Play


In [19]:
# Save to CSV
import os
os.makedirs('data/processed', exist_ok=True)

output_path = 'data/processed/boa_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: data/processed/boa_bank_reviews_clean.csv


In [20]:
print("  PREPROCESSING REPORT — Awash Bank Reviews")

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")



  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2024-01-11  to  2026-05-11

  Rating distribution:
    5 stars :   98 (19.6%)  ███████████████████
    4 stars :   22 ( 4.4%)  ████
    3 stars :   37 ( 7.4%)  ███████
    2 stars :   32 ( 6.4%)  ██████
    1 stars :  311 (62.2%)  ██████████████████████████████████████████████████████████████

  Text length stats:
    Min    : 12 characters
    Median : 88 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source
